In [7]:
import numpy as np
import cv2

In [8]:
# Load Haar Cascade classifier for detecting faces
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

In [9]:
# Read image with alpha channel (transparency)
# cv2.IMREAD_UNCHANGED - Keeps 4 channels (BGR + Alpha)
sunglasses = cv2.imread("subglasses.webp", cv2.IMREAD_UNCHANGED)

In [10]:
sunglasses.shape

(350, 931, 4)

In [ ]:
cap = cv2.VideoCapture(0)
while True:
    flag, frame = cap.read()
    if not flag:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3)
    for x,y,w,h in faces:
        # Resize sunglasses according to face
        overlay_width = w  # width = face width
        overlay_height = int(h * 0.4)  # height = ~40% of face height

        resized_sunglasses = cv2.resize(sunglasses, (overlay_width, overlay_height))
        y_offset = y + h // 4 # roughly eye region
        x_offset = x

        # Handle Transparency (Alpha Blending)
        # Split PNG into color (BGR) and alpha channels
        overlay_img = resized_sunglasses[:, :, :3]
        # Last Channel - alpha (transparency mask)
        mask = resized_sunglasses[:,:,3]

        # Create inverse mask
        mask_inv = cv2.bitwise_not(mask)

        # Define ROI - Region Of Interest on frame
        # This is where we will place the sunglasses
        roi = frame[y_offset:y_offset + overlay_height,
            x_offset:x_offset + overlay_width]

        # Extract background from ROI
        bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
        # Extract foreground
        fg = cv2.bitwise_and(overlay_img, overlay_img, mask=mask)

        combined = cv2.add(bg, fg)
        frame[y_offset:y_offset + overlay_height, 
            x_offset:x_offset + overlay_width] = combined

        cv2.rectangle(frame, (x,y), (x+w, y+h), (0,255,0), 2)
    cv2.imshow("sunglasses", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
pip install dlib


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
     --------- ------------------------------ 0.8/3.3 MB 3.3 MB/s eta 0:00:01
     ---------------------- ----------------- 1.8/3.3 MB 5.0 MB/s eta 0:00:01
     ------------------------------- -------- 2.6/3.3 MB 4.4 MB/s eta 0:00:01
     ---------------------------------------- 3.3/3.3 MB 3.9 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build dlib
Note: you may need to restart the kernel to use u

  error: subprocess-exited-with-error
  
  × Building wheel for dlib (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [102 lines of output]
      INFO:root:running bdist_wheel
      INFO:root:running build
      INFO:root:running build_ext
      Building extension for Python 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
      Invoking CMake setup: 'cmake C:\Users\raian\AppData\Local\Temp\pip-install-yt8jusby\dlib_4b317c2126dd44b1b3e6da45945488e3\tools\python -DCMAKE_LIBRARY_OUTPUT_DIRECTORY=C:\Users\raian\AppData\Local\Temp\pip-install-yt8jusby\dlib_4b317c2126dd44b1b3e6da45945488e3\build\lib.win-amd64-cpython-313 -DDLIB_USE_FFMPEG=OFF -DPYTHON_EXECUTABLE=c:\ProgramData\anaconda3\python.exe -DCMAKE_LIBRARY_OUTPUT_DIRECTORY_RELEASE=C:\Users\raian\AppData\Local\Temp\pip-install-yt8jusby\dlib_4b317c2126dd44b1b3e6da45945488e3\build\lib.win-amd64-cpython-313 -A x64'
      -- Building for: NMake Makefiles
      CMake Error at 

: 

In [ ]:
import dlib

face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
cap = cv2.VideoCapture(0)
while True:
    flag, frame = cap.read()
    if not flag:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3)

    for face in faces:
        landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
        landmarks = landmark_predictor(gray, dlib.rectangle(face[0], face[1], face[0]+face[2], face[1]+face[3]))

        # Get eye points
        left_eye = (landmarks.part(36).x, landmarks.part(36).y)
        right_eye = (landmarks.part(45).x, landmarks.part(45).y)

        # Calculate width between eyes
        eye_width = int(np.linalg.norm(np.array(left_eye) - np.array(right_eye)))

        # Resize sunglasses
        overlay_width = int(eye_width * 2)
        overlay_height = int(overlay_width * 0.5)

        resized_sunglasses = cv2.resize(sunglasses, (overlay_width, overlay_height))

        # Position (center between eyes)
        x_offset = int((left_eye[0] + right_eye[0]) / 2 - overlay_width / 2)
        y_offset = int((left_eye[1] + right_eye[1]) / 2 - overlay_height / 2)

        # Boundary check (VERY IMPORTANT)
        if y_offset < 0 or x_offset < 0:
            continue
        if y_offset + overlay_height > frame.shape[0] or x_offset + overlay_width > frame.shape[1]:
            continue

        # Split overlay
        overlay_img = resized_sunglasses[:, :, :3]
        mask = resized_sunglasses[:, :, 3]
        mask_inv = cv2.bitwise_not(mask)

        roi = frame[y_offset:y_offset + overlay_height,
                    x_offset:x_offset + overlay_width]

        bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
        fg = cv2.bitwise_and(overlay_img, overlay_img, mask=mask)

        combined = cv2.add(bg, fg)
        frame[y_offset:y_offset + overlay_height,
              x_offset:x_offset + overlay_width] = combined

    cv2.imshow("Sunglasses Fit", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
